# Testing Navier Stokes solver.

1. Convert twoD -> threeD. DONE IN `TestingNavierStokes1.ipynb`
2. Add hydrostatic balance. DONE IN THIS NOTEBOOK
3. Add Coriolis force

Based on: `gridap` Tutorial 8: Incompressible Navier-Stokes

twnh June '25

In this tutorial, we will learn
 - How to solve nonlinear multi-field PDEs in Gridap
 - How to build FE spaces whose functions have zero mean value

## Problem statement

The goal of this last tutorial is to solve a nonlinear multi-field PDE. As a model problem, we consider a well known benchmark in computational fluid dynamics, the lid-driven cavity for the incompressible Navier-Stokes equations. Formally, the PDE we want to solve is: find the velocity vector $u$ and the pressure $p$ such that

$$
\left\lbrace
\begin{aligned}
-\Delta u + \nabla p + g \hat{\mathbf k} = 0 &\text{ in }\Omega,\\
\nabla\cdot u = 0 &\text{ in } \Omega,\\
u = g &\text{ on } \partial\Omega,
\end{aligned}
\right.
$$

where the computational domain is the unit square $\Omega \doteq (0,1)^d$, $d=2$. In this example, the driving force is the Dirichlet boundary velocity $g$, which is a non-zero horizontal velocity with a value of $g = (1,0,0)^t$ on the top side of the cavity, namely the boundary $(0,1)\times\{1\}$, and $g=0$ elsewhere on $\partial\Omega$. Since we impose Dirichlet boundary conditions on the entire boundary $\partial\Omega$, the mean value of the pressure is constrained to equal zero,

$$
\int_\Omega q \ {\rm d}\Omega = .
$$

## Numerical Scheme

In order to approximate this problem we chose a formulation based on inf-sub stable $Q_k/P_{k-1}$ elements with continuous velocities and discontinuous pressures. The interpolation spaces are defined as follows.  The velocity interpolation space is

$$
V \doteq \{ v \in [C^0(\Omega)]^d:\ v|_T\in [Q_k(T)]^d \text{ for all } T\in\mathcal{T} \},
$$
where $T$ denotes an arbitrary cell of the FE mesh $\mathcal{T}$, and $Q_k(T)$ is the local polynomial space in cell $T$ defined as the multi-variate polynomials in $T$ of order less or equal to $k$ in each spatial coordinate. Note that, this is the usual continuous vector-valued Lagrangian FE space of order $k$ defined on a mesh of quadrilaterals or hexahedra.  On the other hand, the space for the pressure is

$$
\begin{aligned}
Q_0 &\doteq \{ q \in Q: \  \int_\Omega q \ {\rm d}\Omega = 0\}, \text{ with}\\
Q &\doteq \{ q \in L^2(\Omega):\ q|_T\in P_{k-1}(T) \text{ for all } T\in\mathcal{T}\},
\end{aligned}
$$
where $P_{k-1}(T)$ is the polynomial space of multi-variate polynomials in $T$ of degree less or equal to $k-1$. Note that functions in $Q_0$ are strongly constrained to have zero mean value. This is achieved in the code by removing one degree of freedom from the (unconstrained) interpolation space $Q$ and  adding a constant to the computed pressure so that the resulting function has zero mean value.

The weak form associated to these interpolation spaces reads: find $(u,p)\in U_g \times Q_0$ such that $[r(u,p)](v,q)=0$ for all $(v,q)\in V_0 \times Q_0$
where $U_g$ and $V_0$ are the set of functions in $V$ fulfilling the Dirichlet boundary condition $g$ and $0$  on $\partial\Omega$ respectively. The weak residual $r$ evaluated at a given pair $(u,p)$ is the linear form defined as

$$
[r(u,p)](v,q) \doteq a((u,p),(v,q)),
$$

In order to solve this weak equation with a Newton-Raphson method, one needs to compute the Jacobian associated with the residual $r$. In this case, the Jacobian $j$ evaluated at a pair $(u,p)$ is the bilinear form defined as

$$
[j(u,p)]((\delta u, \delta p),(v,q)) \doteq a((\delta u,\delta p),(v,q))  
$$
The implementation of this numerical scheme is done in Gridap by combining the concepts previously seen for single-field nonlinear PDEs  and linear multi-field problems.

## Discrete model

We start with the discretization of the computational domain. We consider a Cartesian mesh of the unit square.

In [14]:
using Gridap
# n = 20 # works, but 32 was too much
n = 16
domain = (-0.5,0.5,-0.5,0.5,-1,0)
partition = (n,n,n)
model = CartesianDiscreteModel(domain,partition)
gravity = 10.0

10.0

For convenience, we create two new boundary tags,  namely `"diri1"` and `"diri0"`, one for the top side of the square (where the velocity is non-zero), and another for the rest of the boundary (where the velocity is zero).

In [15]:
labels = get_face_labeling(model)

add_tag_from_tags!(labels,"diri1",[22,])
add_tag_from_tags!(labels,"diri0",[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,23,24,25,26])

writevtk(model,"model") ;

## FE spaces

For the velocities, we need to create a conventional vector-valued continuous Lagrangian FE space. In this example, we select a second order interpolation.

In [16]:
D = 2
order = 2
reffeᵤ = ReferenceFE(lagrangian,VectorValue{3,Float64},order)
V = TestFESpace(model,reffeᵤ,conformity=:H1,labels=labels,dirichlet_tags=["diri0","diri1"])

UnconstrainedFESpace()

The interpolation space for the pressure is built as follows

In [17]:
reffeₚ = ReferenceFE(lagrangian,Float64,order-1;space=:P)
Q = TestFESpace(model,reffeₚ,conformity=:L2,constraint=:zeromean)

ZeroMeanFESpace()

With the options `:Lagrangian`, `space=:P`, `valuetype=Float64`, and `order=order-1`, we select the local polynomial space $P_{k-1}(T)$ on the cells $T\in\mathcal{T}$. With the symbol `space=:P` we specifically chose a local Lagrangian interpolation of type "P". Without using `space=:P`, would lead to a local Lagrangian of type "Q" since this is the default for quadrilateral or hexahedral elements. On the other hand, `constraint=:zeromean` leads to a FE space, whose functions are constrained to have mean value equal to zero, which is just what we need for the pressure space. With these objects, we build the trial multi-field FE spaces

In [18]:
uD0 = VectorValue(0,0,0)
uD1 = VectorValue(1e-2,0,0)
khat = VectorValue(0,0,1)
U = TrialFESpace(V,[uD0,uD1])

P = TrialFESpace(Q)

Y = MultiFieldFESpace([V, Q])
X = MultiFieldFESpace([U, P])

MultiFieldFESpace()

## Triangulation and integration quadrature

From the discrete model we can define the triangulation and integration measure

In [19]:
degree = order
Ωₕ = Triangulation(model)
dΩ = Measure(Ωₕ,degree)

GenericMeasure()

The bilinear form reads

In [20]:
a((u,p),(v,q)) = ∫( ∇(v)⊙∇(u) - (∇⋅v)*p + q*(∇⋅u) - gravity*v⋅khat )dΩ

a (generic function with 1 method)

Finally, the Navier-Stokes weak form residual and Jacobian can be defined as

In [21]:
res((u,p),(v,q)) = a((u,p),(v,q))

res (generic function with 1 method)

With the function `res` representing the weak residual, we build the nonlinear FE problem:

In [22]:
op = FEOperator(res,X,Y)

FEOperatorFromWeakForm()

## Nonlinear solver phase

To finally solve the problem, we consider the same nonlinear solver as previously considered for the  $p$-Laplacian equation.

In [23]:
using LineSearches: BackTracking
nls = NLSolver(
  show_trace=true, method=:newton, linesearch=BackTracking())
solver = FESolver(nls)

NonlinearFESolver()

In this example, we solve the problem without providing an initial guess (a default one equal to zero will be generated internally)

In [24]:
uh, ph = solve(solver,op)

Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     7.407407e-04              NaN
     1     1.994854e-12     6.702852e+05


MultiFieldFEFunction():
 num_fields: 2
 num_cells: 4096
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 7358402102681438150

Finally, we write the results for visualization.

In [25]:
writevtk(Ωₕ,"ins-results",cellfields=["uh"=>uh,"ph"=>ph])

(["ins-results.vtu"],)